In [31]:
import os
import random
import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import random_split

from torchvision import transforms

from sklearn.metrics import accuracy_score

from tqdm import tqdm

In [32]:
DATA_DIR = "data/Processed_Train/images"
CSV_PATH = "data/Processed_Train/processed_annotations.csv"

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

In [33]:
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-3 # 0.001
TRAIN_RATIO = 0.8
RANDOM_SEED = 42

In [34]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(DEVICE)

cuda


In [35]:
train_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(15),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

])

In [36]:
valid_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

])

In [37]:
from sklearn.model_selection import train_test_split

annotations = pd.read_csv(CSV_PATH)

train_df, valid_df = train_test_split(
    annotations,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=annotations["Label"]
)

In [38]:
class SnakeDataset(Dataset):

    def __init__(self, annotations, image_dir, transform=None):

        self.annotations = annotations.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):

        row = self.annotations.iloc[index]

        image_path = os.path.join(
            self.image_dir,
            row["filename"]
        )

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = int(row["Label"])

        return image, label

In [39]:
train_dataset = SnakeDataset(
    annotations=train_df,
    image_dir=DATA_DIR,
    transform=train_transform
)

valid_dataset = SnakeDataset(
    annotations=valid_df,
    image_dir=DATA_DIR,
    transform=valid_transform
)

In [40]:
print("Total Images:", len(train_dataset))

Total Images: 4887


In [41]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    pin_memory=torch.cuda.is_available()
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False,
    pin_memory=torch.cuda.is_available()
)

In [42]:
print(len(annotations))
print(len(train_df))
print(len(valid_df))

print(train_df.head())

6109
4887
1222
                                               filename  \
4740  188473055_jpeg.rf.466d7b0f63343cfd34824576ed89...   
409   80818505_jpg.rf.177954c7705d0d0096e7f0d7c628c0...   
2368  127392976_jpeg.rf.7f45516d27de13bbdf1129186c44...   
1482  110829487_jpeg.rf.7e62aa70db88a8a79fb0e7a1ad06...   
1381  38830437_jpeg.rf.42361e17f36bac398f425a92ec2e2...   

                        class  Label  width  height  
4740       ophiophagus hannah      8    237     357  
409        chrysopelea ornata      4    212     500  
2368  trimeresurus albolabris     12    145     204  
1482      dendrelaphis pictus      6    375     402  
1381          daboia russelii      5    421     375  


In [44]:
print(len(train_dataset))
print(len(valid_dataset))

4887
1222


In [45]:
from torchvision.models import efficientnet_b0
from torchvision.models import EfficientNet_B0_Weights

weights = EfficientNet_B0_Weights.DEFAULT

model = efficientnet_b0(weights=weights)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\kisha/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:02<00:00, 8.64MB/s]


In [47]:
NUM_CLASSES = 15

model.classifier[1] = nn.Linear(
    in_features=model.classifier[1].in_features,
    out_features=NUM_CLASSES
)

In [48]:
model = model.to(DEVICE)

In [49]:
print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=15, bias=True)
)


In [50]:
criterion = nn.CrossEntropyLoss()

In [51]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

In [52]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

In [53]:
scaler = torch.amp.GradScaler("cuda")

In [54]:
best_accuracy = 0.0
patience = 5
counter = 0

In [55]:
MODEL_PATH = "models/best_model.pth"